# PowerCo Churn: Cost-Sensitive Classification and Survival Analysis

This notebook builds the full modelling pipeline:

1. Three-way stratified split (train, validation, test).
2. SMOTE-balanced Random Forest inside an `imblearn.Pipeline`.
3. Cost-sensitive threshold selection on the validation fold.
4. 5-fold cross-validation for the cost-optimal threshold.
5. Sensitivity analysis on CLV, campaign cost and retention rate.
6. Model comparison across Logistic Regression, Random Forest and XGBoost.
7. Permutation feature importance on the test fold.
8. Cox proportional-hazards survival model with log-transformed covariates
   and a Schoenfeld residual test of the PH assumption.
9. One-shot evaluation on the test fold.

The shared logic lives under `src/`. The notebook is a thin orchestration
layer.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    average_precision_score, precision_recall_curve, recall_score,
    precision_score, roc_auc_score, roc_curve,
)
from sklearn.inspection import permutation_importance
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

from src.data_loading import load_model_dataset, split_features_target
from src.evaluation import (
    CostMatrix, expected_cost, expected_value,
    optimal_threshold, sweep_thresholds,
)
from src.model import make_smote_rf, three_way_split, cv_threshold

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [ ]:
df = load_model_dataset()
X, y = split_features_target(df)
print(f"Rows: {len(df):,}    Features: {X.shape[1]}    Churn rate: {y.mean():.2%}")

## 1. Train / Validation / Test Split

A 60 / 20 / 20 stratified split. Threshold selection happens on the
validation fold; the test fold is read only once, at the end.

| Fold | Purpose |
|------|---------|
| Train (60%) | Fit the SMOTE + Random Forest pipeline |
| Validation (20%) | Select the cost-optimal threshold |
| Test (20%) | Final evaluation of the chosen policy |

In [ ]:
X_train, X_val, X_test, y_train, y_val, y_test = three_way_split(
    X, y, val_size=0.2, test_size=0.2, random_state=RANDOM_STATE,
)
for name, fold in [("train", y_train), ("val", y_val), ("test", y_test)]:
    print(f"{name:>5}: n = {len(fold):>5}    churn = {fold.mean():.2%}")

## 2. SMOTE + Random Forest Pipeline

`make_smote_rf()` wraps SMOTE inside `imblearn.Pipeline` so the resampler
is refit inside every cross-validation split.

In [ ]:
pipe = make_smote_rf(random_state=RANDOM_STATE)
pipe.fit(X_train, y_train)

y_val_prob = pipe.predict_proba(X_val)[:, 1]
y_test_prob = pipe.predict_proba(X_test)[:, 1]

print(f"Validation AUC-ROC: {roc_auc_score(y_val, y_val_prob):.3f}")
print(f"Test AUC-ROC:       {roc_auc_score(y_test, y_test_prob):.3f}")

## 3. Cost-Sensitive Threshold Selection

The cost function is:

```
expected_cost = FN * 50_000 + FP * 500 + TP * (-10_000)
```

The threshold is selected on the validation fold by minimising this cost.

In [ ]:
costs = CostMatrix(cost_fn=50_000, cost_fp=500, cost_tp=-10_000, cost_tn=0)

t_star, val_cost_star = optimal_threshold(y_val.values, y_val_prob, costs)
val_cost_at_default = expected_cost(y_val.values, y_val_prob, 0.5, costs)

print(f"Optimal threshold (validation): {t_star:.3f}")
print(f"Validation cost at t*       : GBP {val_cost_star:>15,.0f}")
print(f"Validation cost at 0.5      : GBP {val_cost_at_default:>15,.0f}")

In [ ]:
sweep = sweep_thresholds(y_val.values, y_val_prob, costs)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].plot(sweep.thresholds, sweep.costs / 1e6, color='C0')
axes[0].axvline(t_star, color='C3', linestyle='--', label=f't* = {t_star:.2f}')
axes[0].set_xlabel('Threshold')
axes[0].set_ylabel('Expected cost (GBP M, validation)')
axes[0].set_title('Cost vs. threshold')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(sweep.thresholds, sweep.recall, label='Recall', color='C2')
axes[1].plot(sweep.thresholds, sweep.precision, label='Precision', color='C1')
axes[1].axvline(t_star, color='C3', linestyle='--')
axes[1].set_xlabel('Threshold')
axes[1].set_ylabel('Score')
axes[1].set_title('Recall and precision vs. threshold')
axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 4. Cross-Validated Robustness Check

The full pipeline (SMOTE plus Random Forest plus threshold optimisation)
is re-run inside 5-fold stratified cross-validation on the training fold.
The mean and standard deviation of the cost-optimal threshold quantify how
stable the choice is across folds.

In [ ]:
cv = cv_threshold(X_train, y_train, n_splits=5, costs=costs, random_state=RANDOM_STATE)
print(f"CV optimal threshold : {cv.threshold_mean:.3f} +/- {cv.threshold_std:.3f}")
print(f"CV recall @ optimal  : {cv.recall_mean:.3f} +/- {cv.recall_std:.3f}")
print(f"CV cost at optimal   : GBP {cv.cost_mean:>13,.0f} +/- GBP {cv.cost_std:,.0f}")

## 5. Sensitivity Analysis

The cost-optimal threshold depends on three assumed cost parameters: the
cost of missing a churner (CLV), the cost of a retention contact, and the
probability that a retention campaign succeeds. Each one is varied
independently below.

In [ ]:
records = []
for clv in [10_000, 25_000, 50_000, 75_000, 100_000]:
    c = CostMatrix(cost_fn=clv, cost_fp=500, cost_tp=-10_000)
    t, total = optimal_threshold(y_val.values, y_val_prob, c)
    records.append({"CLV (GBP)": clv, "t*": t, "Val cost (GBP)": total})
pd.DataFrame(records)

In [ ]:
records = []
for cc in [100, 250, 500, 1_000, 2_500]:
    c = CostMatrix(cost_fn=50_000, cost_fp=cc, cost_tp=-10_000)
    t, total = optimal_threshold(y_val.values, y_val_prob, c)
    records.append({"Campaign GBP": cc, "t*": t, "Val cost (GBP)": total})
pd.DataFrame(records)

In [ ]:
for ret in [0.10, 0.20, 0.30, 0.40, 0.50]:
    ev = expected_value(y_val.values, y_val_prob, threshold=t_star, retention_rate=ret)
    print(f"retention rate = {ret:.0%}    EV on val fold = GBP {ev:>14,.0f}")

The cost-optimal threshold rises with CLV (fewer false positives are
preferred when missing a churner is cheaper) and rises with campaign cost
(contacts get more expensive). The expected value is positive only when
the retention campaign actually retains around 20% or more of contacted
customers.

## 6. Model Comparison

Three classifiers, all wrapped in an `imblearn.Pipeline` with SMOTE on the
training fold, evaluated at `t*` selected on the validation fold and
scored on the test fold.

In [ ]:
scaler = StandardScaler().fit(X_train)

models = {
    'Logistic Regression': LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
    'Random Forest': RandomForestClassifier(
        n_estimators=100, max_depth=20, min_samples_split=5,
        random_state=RANDOM_STATE, n_jobs=-1),
    'XGBoost': XGBClassifier(
        n_estimators=100, max_depth=6, learning_rate=0.1,
        random_state=RANDOM_STATE, eval_metric='logloss', verbosity=0),
}

rows = []
for name, clf in models.items():
    steps = [('smote', SMOTE(random_state=RANDOM_STATE))]
    if name == 'Logistic Regression':
        steps.append(('scale', StandardScaler()))
    steps.append(('clf', clf))
    pipeline = ImbPipeline(steps=steps)
    pipeline.fit(X_train, y_train)
    p_test = pipeline.predict_proba(X_test)[:, 1]
    y_pred = (p_test >= t_star).astype(int)
    rows.append({
        'Model': name,
        'AUC-ROC (test)': roc_auc_score(y_test, p_test),
        'AUC-PR (test)':  average_precision_score(y_test, p_test),
        f'Recall @ {t_star:.2f}':    recall_score(y_test, y_pred),
        f'Precision @ {t_star:.2f}': precision_score(y_test, y_pred),
    })

results_df = pd.DataFrame(rows)
print(results_df.round(3).to_string(index=False))

AUC-ROC differences of order 0.01 are within bootstrap noise on
n ~ 2,900. Tree-based models outperform Logistic Regression on AUC-PR
(the more informative metric on a 10% positive dataset), indicating
non-linear interactions among predictors.

## 7. Permutation Feature Importance

Computed on the test fold using the frozen Random Forest pipeline.

In [ ]:
perm = permutation_importance(
    pipe, X_test, y_test, n_repeats=10, random_state=RANDOM_STATE, n_jobs=-1,
)
imp = pd.DataFrame({
    'feature': X_test.columns,
    'importance': perm.importances_mean,
    'std': perm.importances_std,
}).sort_values('importance', ascending=False).head(15)

fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(range(len(imp)), imp['importance'][::-1], xerr=imp['std'][::-1], color='steelblue')
ax.set_yticks(range(len(imp)))
ax.set_yticklabels(imp['feature'][::-1])
ax.set_xlabel('Mean decrease in score')
ax.set_title('Permutation importance (test fold, top 15)')
plt.tight_layout(); plt.show()
imp

Anonymised campaign-acquisition codes (`origin_up_*`) and channel codes
(`channel_*`) dominate the importance ranking. Absolute price features
appear but rank below.

This is a classifier-side finding. The Cox model in the next section
uses a different covariate set and supports a different finding.

## 8. Survival Analysis

Churn is reframed as a time-to-event problem. Heavy-tailed covariates are
log-transformed so the hazard ratios are interpretable as percentage
changes. The Schoenfeld residual test checks the proportional-hazards
assumption.

In [ ]:
from src.survival import prepare_cox_frame, fit_cox

covariates = ['cons_12m', 'net_margin', 'var_year_price_off_peak', 'has_gas']
cox_df = prepare_cox_frame(df, covariates=covariates, duration_col='tenure', event_col='churn')
report = fit_cox(cox_df, duration_col='tenure', event_col='churn', penalizer=0.01)

print(f"Concordance index: {report.concordance:.3f}")
print(f"Proportional-hazards assumption holds (all p > 0.05): {report.ph_assumption_holds}\n")
print('Cox summary:')
print(report.summary[['exp(coef)', 'exp(coef) lower 95%', 'exp(coef) upper 95%', 'p']].round(3))
print('\nSchoenfeld residual test:')
print(report.schoenfeld[['test_statistic', 'p']].round(3))

A concordance index of around 0.56 means the Cox model only marginally
separates eventual churners from non-churners on tenure alone. `has_gas`
shows HR around 0.90 (p around 0.04): dual-fuel customers have about 10%
lower churn hazard, statistically significant but moderate in magnitude.

If the Schoenfeld test rejects PH for any covariate, the constant
hazard-ratio interpretation is unreliable for that covariate; a stratified
Cox or a time-varying coefficient model is the appropriate next step.

## 9. Final Test-Fold Evaluation

Reported once, on the test fold, at the threshold selected on the
validation fold.

In [ ]:
baseline = RandomForestClassifier(
    n_estimators=100, max_depth=20,
    random_state=RANDOM_STATE, n_jobs=-1,
)
baseline.fit(X_train, y_train)
p_base = baseline.predict_proba(X_test)[:, 1]
y_pred_base = (p_base >= 0.5).astype(int)

y_pred_final = (y_test_prob >= t_star).astype(int)

baseline_cost = expected_cost(y_test.values, p_base, 0.5, costs)
final_cost    = expected_cost(y_test.values, y_test_prob, t_star, costs)

rows = [
    {'Stage': 'Baseline (RF, t=0.5)',
     'Recall': recall_score(y_test, y_pred_base),
     'Precision': precision_score(y_test, y_pred_base),
     'Test-fold cost (GBP)': baseline_cost},
    {'Stage': f'Final (SMOTE+RF, t={t_star:.2f})',
     'Recall': recall_score(y_test, y_pred_final),
     'Precision': precision_score(y_test, y_pred_final),
     'Test-fold cost (GBP)': final_cost},
]
pd.DataFrame(rows)

In [ ]:
smote_only = make_smote_rf(random_state=RANDOM_STATE).fit(X_train, y_train)
p_smote_only = smote_only.predict_proba(X_test)[:, 1]
smote_only_cost = expected_cost(y_test.values, p_smote_only, 0.5, costs)

gain_smote = baseline_cost - smote_only_cost
gain_threshold = smote_only_cost - final_cost
ratio = gain_threshold / gain_smote if gain_smote else float('inf')

print(f"Baseline cost      : GBP {baseline_cost:>13,.0f}")
print(f"+ SMOTE only       : GBP {smote_only_cost:>13,.0f}    (delta GBP {gain_smote:>+13,.0f})")
print(f"+ Threshold opt    : GBP {final_cost:>13,.0f}    (delta GBP {gain_threshold:>+13,.0f})")
print(f"\nThreshold optimisation contributed {ratio:.1f}x the cost reduction of SMOTE alone (test fold).")

## Limitations

1. The data is a single 2015 snapshot. The test-fold cost reduction is a
   one-shot estimate, not an annualised figure, and there is no
   out-of-time validation.
2. The cost parameters (CLV £50k, campaign £500, retention benefit £10k)
   are assumed values. The sensitivity sweep above is the right way to
   express how outputs respond to those assumptions.
3. SMOTE rebalances the training class only; the test fold remains at the
   natural prevalence. Probability calibration after SMOTE is a known
   issue and would need isotonic or Platt scaling before deployment.
4. The dataset shows no evidence that absolute price level is a primary
   driver of churn in this snapshot. Price changes, competitor pricing
   and contract-change history are not present in the data, so this is
   not a refutation of price sensitivity in general.
5. A threshold of around 0.05 contacts a large fraction of the customer
   base. This is an upper bound on recall under the assumed cost matrix;
   any production deployment would need contact-fatigue constraints that
   are not modelled here.